In [1]:
import pandas as pd
from tqdm.notebook import tqdm
from urlextract import URLExtract

extractor = URLExtract()

tqdm.pandas()

In [2]:
%%time
# data file can be downloaded from here - https://x.com/i/communitynotes/download-data
notes_file_path = (
    r"/path/to//notes.tsv"
)
notes_df = pd.read_csv(notes_file_path, sep="\t")

<timed exec>:2: DtypeWarning: Columns (5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.


CPU times: user 13.3 s, sys: 1.19 s, total: 14.5 s
Wall time: 14.5 s


In [3]:
rows, cols = notes_df.shape
print(f"{rows:,} rows × {cols:,} columns")

1,855,148 rows × 23 columns


In [4]:
notes_df.columns

Index(['noteId', 'noteAuthorParticipantId', 'createdAtMillis', 'tweetId',
       'classification', 'believable', 'harmful', 'validationDifficulty',
       'misleadingOther', 'misleadingFactualError',
       'misleadingManipulatedMedia', 'misleadingOutdatedInformation',
       'misleadingMissingImportantContext', 'misleadingUnverifiedClaimAsFact',
       'misleadingSatire', 'notMisleadingOther',
       'notMisleadingFactuallyCorrect',
       'notMisleadingOutdatedButNotWhenWritten', 'notMisleadingClearlySatire',
       'notMisleadingPersonalOpinion', 'trustworthySources', 'summary',
       'isMediaNote'],
      dtype='object')

In [5]:
num_duplicates = notes_df.duplicated(subset=["noteId"]).sum()
print(f"Number of duplicate values: {num_duplicates}")

Number of duplicate values: 0


In [6]:
num_duplicates = notes_df.duplicated(subset=["summary"]).sum()
print(f"Number of duplicate values: {num_duplicates:,}")

Number of duplicate values: 200,320


In [7]:
notes_df["summary"].head()

0    The House failed to pass a border protection l...
1    The United States has 50 States     https://da...
2    TikTok only mentions “ban” and chooses to igno...
3    This could be considered a threat     https://...
4    Forbes has a good rundown of the investigation...
Name: summary, dtype: object

In [8]:
def extract_urls_with_urlextract(text):
    if pd.isna(text):
        return []
    return extractor.find_urls(str(text))

In [11]:
%%time
notes_df["summary_urls"] = notes_df["summary"].progress_apply(
    extract_urls_with_urlextract
)

  0%|          | 0/1855148 [00:00<?, ?it/s]

CPU times: user 55min 11s, sys: 10.4 s, total: 55min 22s
Wall time: 55min 20s


In [12]:
notes_df.to_csv(r"notes_with_extracted_urls.csv", index=False)

In [13]:
unique_lengths = notes_df["summary_urls"].apply(len).unique()
print(sorted(unique_lengths))

[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(44), np.int64(46), np.int64(47), np.int64(49), np.int64(134)]


In [10]:
# row_with_12_urls = notes_df[notes_df['summary_urls'].apply(len) == 12]
# row_with_12_urls[['summary', 'summary_urls']].head(5).to_csv('temp.csv', index=False)